# AiJockey — Train Technique Classifier (Tier 1)

Replaces the rule-based decision tree in `planner.transition_score` with a
small MLP trained on synthetic transition data.

**Pipeline**:
1. Run env install + analyze your clips (assumes you've already done this)
2. Build synthetic dataset: render every technique on every clip pair, score smoothness
3. Train MLP classifier on (features -> best technique)
4. Plug classifier into pipeline via `--classifier` flag

**Time on T4**: dataset build ~5-10 min for 5 clips, training ~2 min.

## 1. Setup (skip if already done in pipeline notebook)

In [ ]:
!nvidia-smi | head -10
!apt-get install -y rubberband-cli ffmpeg > /dev/null 2>&1
!git clone https://github.com/architagrawal/aiJockey.git 2>/dev/null || (cd aiJockey && git pull)
%cd aiJockey

In [ ]:
# Install deps (Py 3.12 compatible set)
!pip install --force-reinstall --no-deps "numpy<2.0"
!pip install --no-build-isolation --upgrade "cython<3"
!pip install demucs==4.0.1 librosa==0.10.2 soundfile==0.12.1 \
    pyrubberband pyloudnorm==0.1.1 scipy tqdm transformers
!pip install --no-build-isolation git+https://github.com/CPJKU/madmom.git
print('install done')

## 2. Verify analyzed clips exist

If you haven't analyzed yet, run analyze first via pipeline notebook.

In [ ]:
import os
if not os.path.exists('cache') or not [f for f in os.listdir('cache') if f.endswith('.json')]:
    print('NO ANALYZED CLIPS in cache/. Run analyze first:')
    print('  !python src/main.py analyze --clips clips/ --device cuda')
else:
    print('analyzed clips:')
    !ls cache/*.json 2>/dev/null | wc -l

## 3. Build synthetic dataset

For each clip pair, render every transition technique, score smoothness, label by best.

5 clips → ~20 ordered pairs × 15 techniques = ~300 renders, ~5-10 min on T4.

In [ ]:
!python src/training/synthetic_dataset.py \
    --cache cache/ \
    --samples samples/ \
    --out datasets/synthetic_transitions.npz

In [ ]:
import numpy as np
d = np.load('datasets/synthetic_transitions.npz', allow_pickle=True)
print('X:', d['X'].shape, 'y:', d['y'].shape, 'scores:', d['scores'].shape)
print('mean smoothness score:', float(d['scores'].mean()))
import collections
techs = list(d['techniques'])
counts = collections.Counter(d['y'])
for idx, c in sorted(counts.items()):
    print(f"  {techs[idx]:20s} {c}")

## 4. Train classifier

In [ ]:
!python src/training/classifier.py \
    --dataset datasets/synthetic_transitions.npz \
    --ckpt checkpoints/technique_classifier.pt \
    --epochs 100 \
    --batch_size 16 \
    --lr 1e-3

## 5. Plot training history

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open('checkpoints/technique_classifier.history.json'))
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(h['train_loss'], label='train'); ax[0].plot(h['val_loss'], label='val')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].set_title('loss')
ax[1].plot(h['val_acc']); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('val acc'); ax[1].set_title('accuracy')
plt.tight_layout(); plt.show()

## 6. Generate mix using trained classifier

In [ ]:
!python src/main.py plan \
    --cache cache/ \
    --duration 300 \
    --classifier checkpoints/technique_classifier.pt \
    --max_clips 50

!python src/main.py execute --timeline output/timeline.json --cache cache/ --out output/raw_mix_ml.wav
!python src/main.py master --in_path output/raw_mix_ml.wav --out output/final_mix_ml.wav --lufs -9

In [ ]:
from IPython.display import Audio
Audio('output/final_mix_ml.wav')

## 7. A/B compare with rule-tree version

Run plan WITHOUT --classifier flag, render, listen.
Compare against the ML version above.

In [ ]:
!python src/main.py plan --cache cache/ --duration 300 --max_clips 50
!python src/main.py execute --timeline output/timeline.json --cache cache/ --out output/raw_mix_rules.wav
!python src/main.py master --in_path output/raw_mix_rules.wav --out output/final_mix_rules.wav --lufs -9

from IPython.display import Audio, display
print('rule-tree mix:'); display(Audio('output/final_mix_rules.wav'))
print('ML classifier mix:'); display(Audio('output/final_mix_ml.wav'))

## 8. Persist trained model + dataset to Drive

So you don't lose them when Colab session ends.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil, os
# os.makedirs('/content/drive/MyDrive/AiJockey_models', exist_ok=True)
# shutil.copy('checkpoints/technique_classifier.pt', '/content/drive/MyDrive/AiJockey_models/')
# shutil.copy('datasets/synthetic_transitions.npz', '/content/drive/MyDrive/AiJockey_models/')
# print('saved to Drive')

## What's next

- **More clips**: 5 -> 20+ for better dataset diversity (analyze them first)
- **Better labels**: replace synthetic smoothness score with real DJ mix labels (Tier 2)
- **Bigger model**: TechniqueClassifier hidden dims, more epochs
- **Multi-task**: predict bars/params alongside technique name
- **RLHF**: rate generated mixes pairwise (you vs Spotify auto-DJ), train preference model

Iterate: collect more clips, re-build dataset, re-train classifier. Each iteration improves the pipeline.

**Ownership**: `checkpoints/technique_classifier.pt` is your trained model. Weights are yours under AGPL-3.0. No HF dependency for inference path (unlike CLAP for analysis).